1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure local package imports work from any notebook launch directory.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.processor import DataProcessor
from src.models import EnsembleWrapper
from src.conformal import get_residuals, apply_enbpi, rolling_agaci
from src.visualization import calculate_metrics

import yaml

2. Load Config

In [ ]:
with open("../configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

np.random.seed(config["seed"])

3. Load Data

In [ ]:
dp = DataProcessor(window=config["data"]["window_size"])

data_path = "../" + config["data"]["file_path"]
series = dp.load_series(data_path, column=config["data"]["target_column"])

X, y = dp.create_sequences(series)

(X_train, y_train), (X_cal, y_cal), (X_test, y_test) = dp.split_data(
    X, y,
    train_p=config["data"]["train_split"],
    cal_p=config["data"]["cal_split"]
)

dp.fit_scaler(y_train)
X_train = dp.transform(X_train)
y_train = dp.transform(y_train)
X_cal = dp.transform(X_cal)
y_cal = dp.transform(y_cal)
X_test = dp.transform(X_test)
y_test = dp.transform(y_test)

# Reshape X to 3-D (samples, timesteps, features) required by LSTM/GRU
X_train = X_train[..., np.newaxis]
X_cal   = X_cal[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

🔬 EXPERIMENT 1 — Effect of α (coverage vs width)
🟩 4. Alpha Sensitivity

In [ ]:
alphas = [0.05, 0.1, 0.15, 0.2]

results_alpha = []

model = EnsembleWrapper("LSTM", B=5)
model.fit(X_train, y_train, epochs=10)

pred_cal = model.predict(X_cal)
pred_test = model.predict(X_test)

residuals = get_residuals(y_cal, pred_cal)

for alpha in alphas:

    l_enbpi, u_enbpi = apply_enbpi(pred_test, residuals, alpha)
    l_agaci, u_agaci = rolling_agaci(pred_test, y_test, residuals, alpha)

    enbpi_cov, enbpi_width = calculate_metrics(y_test, l_enbpi, u_enbpi)
    agaci_cov, agaci_width = calculate_metrics(y_test, l_agaci, u_agaci)

    results_alpha.append({
        "alpha": alpha,
        "EnbPI_Coverage": enbpi_cov,
        "EnbPI_Width": enbpi_width,
        "AgACI_Coverage": agaci_cov,
        "AgACI_Width": agaci_width,
    })

df_alpha = pd.DataFrame(results_alpha)
df_alpha

📉 5. Plot α Sensitivity

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(df_alpha["alpha"], df_alpha["EnbPI_Coverage"], marker="o", label="EnbPI Coverage")
plt.plot(df_alpha["alpha"], df_alpha["AgACI_Coverage"], marker="o", label="AgACI Coverage")

plt.axhline(1 - 0.1, linestyle="--", color="black", label="Target Coverage (0.9)")

plt.title("Coverage vs Alpha")
plt.xlabel("Alpha")
plt.ylabel("Coverage")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

🔬 EXPERIMENT 2 — Rolling Window Sensitivity (AgACI)
🟩 6. Window Sensitivity

In [ ]:
windows = [10, 25, 50, 100]

results_window = []

for w in windows:

    l, u = rolling_agaci(pred_test, y_test, residuals, window=w)

    cov, width = calculate_metrics(y_test, l, u)

    results_window.append({
        "window": w,
        "coverage": cov,
        "width": width
    })

df_window = pd.DataFrame(results_window)
df_window

📉 7. Plot Window Sensitivity

In [ ]:
fig, ax1 = plt.subplots(figsize=(10,5))

ax1.plot(df_window["window"], df_window["coverage"], marker="o", color="blue")
ax1.set_ylabel("Coverage", color="blue")
ax1.set_xlabel("Window Size")

ax2 = ax1.twinx()
ax2.plot(df_window["window"], df_window["width"], marker="o", color="red")
ax2.set_ylabel("Interval Width", color="red")

plt.title("AgACI Sensitivity to Window Size")
plt.grid(alpha=0.3)
plt.show()

🔬 EXPERIMENT 3 — Ensemble Size Sensitivity
🟩 8. Ensemble Size Test

In [ ]:
ensemble_sizes = [1, 3, 5, 10]

results_ens = []

for B in ensemble_sizes:

    model = EnsembleWrapper("LSTM", B=B)
    model.fit(X_train, y_train, epochs=10)

    pred = model.predict(X_test)

    residuals = get_residuals(y_cal, model.predict(X_cal))

    l, u = apply_enbpi(pred, residuals)

    cov, width = calculate_metrics(y_test, l, u)

    results_ens.append({
        "B": B,
        "coverage": cov,
        "width": width
    })

df_ens = pd.DataFrame(results_ens)
df_ens

📉 9. Plot Ensemble Effect

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(df_ens["B"], df_ens["coverage"], marker="o", label="Coverage")
plt.plot(df_ens["B"], df_ens["width"], marker="o", label="Width")

plt.title("Ensemble Size Sensitivity")
plt.xlabel("Number of Models (B)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

📌 10. Summary Table (Thesis-ready)

In [ ]:
summary = pd.concat([
    df_alpha.assign(experiment="alpha"),
    df_window.assign(experiment="window"),
    df_ens.assign(experiment="ensemble")
])

summary